# Experiment 17 — Theme Business Needs + Description + Stage — Individual Calls

No record key is sent to the model.

## What the LLM sees

Theme Business Needs + Theme Description + Stage data + candidate L3s.

**Not sent:** record key, ground truth, L1/L2 hierarchy.

## Configuration and data

In [ ]:
from pathlib import Path
from time import perf_counter
import ast, json, os
import pandas as pd
from IPython.display import display
from common import call_llm_with_metrics, load_gateway, parse_json_response, save_results_excel, score_sets

HERE=Path.cwd()
def resolve(name, env):
    if os.getenv(env): return Path(os.environ[env]).expanduser()
    for root in (HERE,HERE.parent,HERE/'l3_experiments'):
        p=root/name
        if p.exists(): return p
    raise FileNotFoundError(name)
DATASET_PATH=resolve('golden_valid_set.parquet','L3_GOLDEN_VALID_SET_PATH')
GT_PATH=resolve('results/epic_l3_ground_truth_full_golden.xlsx','L3_GROUND_TRUTH_PATH')
SAMPLE_SIZE=50
SAMPLE_SEED=42

EXPERIMENT_NAME='E17_THEME_NEEDS_DESCRIPTION_STAGE_INDIVIDUAL_NO_RECORD_KEY'


## Load combined valid set and Jira GT

In [ ]:
def clean(v):
    if v is None: return ''
    try:
        if pd.isna(v): return ''
    except (TypeError,ValueError): pass
    return str(v).strip()

def parsed(v):
    if v is None: return None
    if hasattr(v,'tolist') and not isinstance(v,(str,bytes)): v=v.tolist()
    if isinstance(v,(dict,list,tuple,set)): return v
    try:
        if pd.isna(v): return None
    except (TypeError,ValueError): pass
    s=str(v).strip()
    if not s: return None
    for fn in (json.loads,ast.literal_eval):
        try: return fn(s)
        except Exception: pass
    return s

def as_list(v):
    v=parsed(v)
    if v is None: return []
    if isinstance(v,(list,tuple,set)): return [clean(x) for x in v if clean(x)]
    return [clean(v)] if clean(v) else []

def col(df,*names,required=True):
    lower={str(c).lower():c for c in df.columns}
    for n in names:
        if n in df.columns: return n
        if n.lower() in lower: return lower[n.lower()]
    if required: raise KeyError(f'Missing column; expected one of {names}')

def gt_map():
    g=pd.read_excel(GT_PATH,sheet_name='jira_l3_ground_truth',dtype=str)
    k,l=col(g,'epic_key'),col(g,'l3_capability_id')
    s=col(g,'status',required=False)
    if s: g=g[g[s].fillna('').str.lower().eq('ok')]
    g=g[g[l].notna()].copy(); g[l]=g[l].map(clean)
    return {clean(k0):sorted(set(x[l])) for k0,x in g.groupby(k,sort=False)}

def load_population():
    f=pd.read_parquet(DATASET_PATH)
    req={
      'theme_key':col(f,'theme_key','key'),'record_key':col(f,'epic_key'),
      'stage_ids':col(f,'stage_ids'),'candidate_l3_ids':col(f,'candidate_l3_ids'),
      'theme_description':col(f,'theme_description','description'),
      'theme_business_needs':col(f,'theme_business_needs','businessNeeds')}
    opt={
      'stage_data':col(f,'stage_data','value_stream_stages','stage_contexts',required=False),
      'candidate_l3_capabilities':col(f,'candidate_l3_capabilities','candidate_l3s',required=False),
      'candidate_l3_by_stage':col(f,'candidate_l3_by_stage','stage_candidate_l3_ids','stage_candidate_l3_capabilities',required=False)}
    p=pd.DataFrame({k:f[v] for k,v in req.items()})
    for k,v in opt.items(): p[k]=f[v] if v else None
    for k in ('theme_key','record_key','theme_description','theme_business_needs'): p[k]=p[k].map(clean)
    p=p.drop_duplicates(['theme_key','record_key']).sort_values(['theme_key','record_key']).reset_index(drop=True)
    if len(p)<SAMPLE_SIZE: raise ValueError(f'Need {SAMPLE_SIZE} rows, found {len(p)}')
    p=p.sample(SAMPLE_SIZE,random_state=SAMPLE_SEED,replace=False)
    gm=gt_map(); p['gt_l3_ids']=p['record_key'].map(gm)
    miss=p[p.gt_l3_ids.isna()].record_key.tolist()
    if miss: raise ValueError(f'GT workbook missing sampled records: {miss}')
    return p.sort_values(['theme_key','record_key']).reset_index(drop=True)

evaluation_population=load_population()
print(f'Selected {len(evaluation_population)} rows from golden_valid_set.parquet with seed={SAMPLE_SEED} across {evaluation_population.theme_key.nunique()} Themes.')
display(evaluation_population.head(50))


## Build Stage payloads

In [ ]:
STAGE_PATH = resolve("VSSrv.csv", "L3_STAGE_PATH")
STAGE_CAPABILITY_MAP_PATH = resolve(
    "VSSCaprv (1).csv",
    "L3_STAGE_CAPABILITY_MAP_PATH",
)

stage_frame = pd.read_csv(
    STAGE_PATH,
    dtype=str,
    encoding="cp1252",
    encoding_errors="replace",
)
stage_capability_map = pd.read_csv(
    STAGE_CAPABILITY_MAP_PATH,
    dtype=str,
    encoding="cp1252",
    encoding_errors="replace",
)


def stage_context(stage_id):
    """Return the full governed metadata for one Value Stream Stage."""
    stage_id = clean(stage_id)
    match = stage_frame.loc[
        stage_frame["Value Stream Stage ID"].astype(str).str.strip().eq(stage_id)
    ]

    if match.empty:
        raise KeyError(f"No Stage metadata found for {stage_id}")

    row = match.iloc[0]
    return {
        "stage_id": stage_id,
        "stage_name": clean(row["Value Stream Stage Name"]),
        "stage_description": clean(row["Value Stream Stage Description"]),
        "entrance_criteria": clean(row["Value Stream Stage Entrance Criteria"]),
        "exit_criteria": clean(row["Value Stream Stage Exit Criteria"]),
    }


def candidate_rows_for_stage(stage_id, allowed_candidate_ids):
    """Return full governed L3 metadata for candidates belonging to one Stage."""
    stage_id = clean(stage_id)
    rows = stage_capability_map.loc[
        stage_capability_map["Value Stream Stage ID"]
        .astype(str)
        .str.strip()
        .eq(stage_id)
    ].copy()

    allowed = {clean(value) for value in allowed_candidate_ids if clean(value)}
    if allowed:
        rows = rows.loc[
            rows["Capability ID"].astype(str).str.strip().isin(allowed)
        ]

    rows = (
        rows
        .drop_duplicates(subset=["Capability ID"], keep="first")
        .sort_values(["Capability Name", "Capability ID"], kind="stable")
    )

    return [
        {
            "capability_id": clean(row["Capability ID"]),
            "capability_name": clean(row["Capability Name"]),
            "capability_description": clean(row["Capability Description"]),
            "capability_tier": clean(row["Capability Tier"]),
        }
        for _, row in rows.iterrows()
    ]


def row_stages(row):
    """Build Stage-specific prompt payloads for one evaluation record."""
    stage_ids = as_list(row["stage_ids"])
    allowed_candidate_ids = as_list(row["candidate_l3_ids"])

    return [
        {
            "stage": stage_context(stage_id),
            "candidates": candidate_rows_for_stage(
                stage_id,
                allowed_candidate_ids,
            ),
        }
        for stage_id in stage_ids
    ]


def merged_stages(rows):
    """Build one deduplicated Stage payload per Theme batch."""
    stage_to_allowed = {}

    for row in rows.to_dict("records"):
        allowed_candidate_ids = as_list(row["candidate_l3_ids"])
        for stage_id in as_list(row["stage_ids"]):
            stage_to_allowed.setdefault(stage_id, set()).update(
                allowed_candidate_ids
            )

    return [
        {
            **stage_context(stage_id),
            "candidate_l3_capabilities": candidate_rows_for_stage(
                stage_id,
                stage_to_allowed[stage_id],
            ),
        }
        for stage_id in sorted(stage_to_allowed)
    ]


## Production prompt

In [ ]:
SYSTEM_PROMPT = """\
You are performing Level 3 business capability classification.

An L3 capability is a Level 3 business capability: a specific business function within the enterprise capability hierarchy.

Use Theme Business Needs as the primary business evidence.
Use Theme Description as supporting context that can clarify the Business Needs, but do not use it to introduce unsupported business functions.
Use the supplied Value Stream Stage data to determine the relevant business boundary.
Select every supplied candidate L3 capability whose business function is directly supported by the supplied Theme context within that Stage boundary.

EVIDENCE

Theme Business Needs describes the business outcomes, requirements, and functions that need to be delivered.
Theme Description provides supporting scope and intent for those Business Needs.
Stage data provides the governed business-process boundary for this classification.

For each candidate L3:
- capability_id is the exact identifier to return when selected.
- capability_description is the primary semantic definition of the business function.
- capability_name is a supporting business label.
- capability_tier is supporting taxonomy context only.

Do not infer business meaning from capability_id.

CLASSIFICATION

1. Identify the business functions directly supported by the Theme context.
2. Use the Stage name, description, entrance criteria, and exit criteria to constrain those functions to the supplied Stage boundary.
3. Compare that evidence against the supplied candidate L3 capability definitions.
4. Select every candidate whose business function is directly supported.

Do not select a capability merely because:
- it belongs to the supplied Stage,
- it shares terminology with the Theme context,
- it is broadly related,
- it is upstream or downstream,
- it commonly supports another capability.

Only return capability_id values supplied in Candidate L3 Capabilities.
If no candidate is supported, return an empty list.

OUTPUT

Return JSON only:

{"l3":["CAP00000123","CAP00000456"]}

Do not return reasons, explanations, Markdown, or additional fields.
"""


def _prompt_text(value):
    return "" if value is None else str(value).strip()


def _format_stage_block(stage, candidate_rows, label="Stage Data:"):
    lines = [
        label,
        f"  Stage ID: {_prompt_text(stage.get('stage_id'))}",
        f"  Stage Name: {_prompt_text(stage.get('stage_name'))}",
        f"  Stage Description: {_prompt_text(stage.get('stage_description'))}",
        f"  Entrance Criteria: {_prompt_text(stage.get('entrance_criteria'))}",
        f"  Exit Criteria: {_prompt_text(stage.get('exit_criteria'))}",
        "",
        "Candidate L3 Capabilities:",
    ]

    if not candidate_rows:
        lines.append("  None")
        return "\n".join(lines)

    for index, capability in enumerate(candidate_rows, start=1):
        lines.extend(
            [
                f"  {index}.",
                f"    Capability ID: {_prompt_text(capability.get('capability_id'))}",
                f"    Capability Name: {_prompt_text(capability.get('capability_name'))}",
                f"    Capability Description: {_prompt_text(capability.get('capability_description'))}",
                f"    Capability Tier: {_prompt_text(capability.get('capability_tier'))}",
            ]
        )

    return "\n".join(lines)


def build_user_prompt(context, stage, candidate_rows):
    sections = [
        "Theme Business Needs:",
        _prompt_text(context.get("theme_business_needs")),
    ]

    sections.extend([
        "",
        "Theme Description:",
        _prompt_text(context.get("theme_description")),
    ])

    sections.extend(
        [
            "",
            _format_stage_block(stage, candidate_rows),
        ]
    )

    return "\n".join(sections).strip()


## Prompt preview — exactly as sent

In [ ]:
r=evaluation_population.iloc[0].to_dict(); p=row_stages(r)[0]
preview_user_prompt=build_user_prompt(theme_context(r),p['stage'],p['candidates'])
print('SYSTEM PROMPT — EXACT TEXT SENT TO LLM\n'+'='*80+'\n'+SYSTEM_PROMPT+'\n\nUSER PROMPT — EXACT TEXT SENT TO LLM\n'+'='*80+'\n'+preview_user_prompt)

## Prediction and evaluation

In [ ]:
def validate(payload,allowed):
    if not isinstance(payload,dict) or set(payload)!={'l3'} or not isinstance(payload['l3'],list): raise ValueError('Expected {"l3": [...]}')
    out=[]; seen=set(); allowed=set(allowed)
    for cid in payload['l3']:
        if not isinstance(cid,str) or cid.strip() not in allowed or cid.strip() in seen: raise ValueError(f'Invalid L3: {cid}')
        cid=cid.strip(); seen.add(cid); out.append(cid)
    return out

def predict(gateway,ctx,p):
    u=build_user_prompt(ctx,p['stage'],p['candidates'])
    if not p['candidates']: return [],None
    raw,m=call_llm_with_metrics(gateway,SYSTEM_PROMPT,u,id='9zdn8n',reasoning_effort='low')
    return validate(parse_json_response(raw),[c['capability_id'] for c in p['candidates']]),m

def run():
    g=load_gateway(); rr=[]; cc=[]
    for r in evaluation_population.to_dict('records'):
        pred=set(); status='ok'; err=None
        for p in row_stages(r):
            sid=p['stage']['stage_id']; t=perf_counter()
            try:
                vals,m=predict(g,theme_context(r),p); pred.update(vals)
                cc.append({'experiment':EXPERIMENT_NAME,'theme_key':r['theme_key'],'record_key':r['record_key'],'stage_id':sid,'status':'ok','latency_seconds':m.get('latency_seconds') if m else 0,'input_tokens':m.get('input_tokens') if m else 0,'output_tokens':m.get('output_tokens') if m else 0,'total_tokens':m.get('total_tokens') if m else 0,'error':None})
            except Exception as e:
                status='error'; err=str(e); cc.append({'experiment':EXPERIMENT_NAME,'theme_key':r['theme_key'],'record_key':r['record_key'],'stage_id':sid,'status':'error','latency_seconds':perf_counter()-t,'input_tokens':None,'output_tokens':None,'total_tokens':None,'error':err}); break
        truth=r['gt_l3_ids']; scores=score_sets(sorted(pred),truth) if status=='ok' else {'exact_match':None,'precision':None,'recall':None,'f1':None,'predicted_count':None,'truth_count':len(truth)}
        rr.append({'experiment':EXPERIMENT_NAME,'theme_key':r['theme_key'],'record_key':r['record_key'],'stage_ids':as_list(r['stage_ids']),'predicted_l3_ids':sorted(pred) if status=='ok' else None,'gt_l3_ids':truth,'status':status,'error':err,**scores})
    return pd.DataFrame(rr),pd.DataFrame(cc)

results,call_metrics=run(); scored=results[results.status.eq('ok')]; calls=call_metrics[call_metrics.status.eq('ok')]
summary=pd.DataFrame([{'scope':'fixed_50_from_golden_valid_set_seed_42','evaluated_records':len(scored),'exact_match_accuracy':scored.exact_match.mean() if len(scored) else 0,'mean_precision':scored.precision.mean() if len(scored) else 0,'mean_recall':scored.recall.mean() if len(scored) else 0,'mean_f1':scored.f1.mean() if len(scored) else 0}])
latency_tokens=pd.DataFrame([{'successful_calls':len(calls),'failed_calls':int(call_metrics.status.eq('error').sum()),'avg_latency_seconds':calls.latency_seconds.mean() if len(calls) else None,'p50_latency_seconds':calls.latency_seconds.quantile(.5) if len(calls) else None,'p95_latency_seconds':calls.latency_seconds.quantile(.95) if len(calls) else None,'total_input_tokens':calls.input_tokens.sum() if len(calls) else 0,'total_output_tokens':calls.output_tokens.sum() if len(calls) else 0,'total_tokens':calls.total_tokens.sum() if len(calls) else 0,'tokens_per_scored_record':calls.total_tokens.sum()/len(scored) if len(scored) else None}])
display(summary); display(latency_tokens); display(call_metrics); display(results.head(50))
print('Saved',save_results_excel(results,EXPERIMENT_NAME,'results',extra_sheets={'evaluation_summary':summary,'llm_metrics':call_metrics,'latency_tokens':latency_tokens,'evaluation_population':evaluation_population}))
